# 01 Standardize and Chunk

这个 Notebook 的第一个目标不是直接切块，而是先把之前已经解析好的财报全部标准化。

当前仓库里真正可用的输入源是：

- `Report_Crawer/03_Data_Processor/US/parsed_filings/*.json`
- `Report_Crawer/03_Data_Processor/CN/parsed_filings/*.json`

US 和 CN 的 `meta_data` 结构不同，所以不能直接切 chunk。正确顺序应该是：

1. 读取所有 parsed filings
2. 统一标准化成同一种文档结构
3. 把标准化结果先保存下来
4. 下一步再从标准化结果出发做 chunk


## 第 1 步：准备路径和输出目录

这一部分只负责三件事：

- 找到 US/CN parsed filings 的目录
- 创建 `04_Embedding/data` 输出目录
- 定义标准化后的输出文件路径


In [1]:
from __future__ import annotations

import hashlib
import html
import json
import re
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional

BASE_DIR = Path('/Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer')
US_PARSED_DIR = BASE_DIR / '03_Data_Processor' / 'US' / 'parsed_filings'
CN_PARSED_DIR = BASE_DIR / '03_Data_Processor' / 'CN' / 'parsed_filings'

OUTPUT_DIR = BASE_DIR / '04_Embedding' / 'chunked_data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STANDARDIZED_JSONL = OUTPUT_DIR / 'standardized_filings.jsonl'

print('US parsed dir:', US_PARSED_DIR)
print('CN parsed dir:', CN_PARSED_DIR)
print('Output dir:', OUTPUT_DIR)
print('Standardized output:', STANDARDIZED_JSONL)

US parsed dir: /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/03_Data_Processor/US/parsed_filings
CN parsed dir: /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/03_Data_Processor/CN/parsed_filings
Output dir: /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/04_Embedding/chunked_data
Standardized output: /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/04_Embedding/chunked_data/standardized_filings.jsonl


## 第 2 步：定义标准化后的统一文档结构

这里先把后面所有步骤都要依赖的统一 schema 定死。

标准化后的字段包括：

- `market`
- `company_code`
- `ticker`
- `company_name`
- `report_year`
- `filing_date`
- `document_type`
- `title`
- `language`
- `source_path`
- `source_sha256`
- `sections`

这样做的好处是，后面做 chunk、embedding、检索时，完全不需要再区分 US 和 CN 原始字段名。

In [2]:
@dataclass(slots=True)
class StandardizedDocument:
    market: str
    company_code: str
    ticker: Optional[str]
    company_name: str
    report_year: int
    filing_date: Optional[str]
    document_type: str
    title: str
    language: str
    source_path: str
    source_sha256: str
    sections: dict[str, str]

## 第 3 步：写基础清洗函数

标准化之前，先把文本做基础清洗。

这里主要处理：

- HTML 实体，例如 `&#160;`、`&#8217;`
- 多余空格和换行
- 空 section
- 计算源文件哈希，方便后面判断文件有没有变化


In [3]:
def compute_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
#用hashlib 给财报做一个指纹，保证没有变化


def clean_text(text: str) -> str:
    text = html.unescape(text or '')#把转义符复原
    text = text.replace('\xa0', ' ')
    text = text.replace('\r\n', '\n').replace('\r', '\n')#把所有换行都变成/n
    text = re.sub(r'[ \t]+', ' ', text)#替换连续空格
    text = re.sub(r'\n{3,}', '\n\n', text) #讲多余的空行变成2行
    return text.strip()


def normalize_sections(raw_sections: dict) -> dict[str, str]:
    """Process and normalize raw filing sections.

    Args:
        raw_sections: A dictionary of raw section data loaded from a filing.

    Returns:
        A dictionary mapping cleaned section titles to normalized section text.
    """
    out: dict[str, str] = {}
    for key, value in (raw_sections or {}).items():
        if not isinstance(value, str):
            continue
        text = clean_text(value)
        if text:
            out[str(key).strip()] = text
    return out

## 第 4 步：分别实现 US 和 CN 的标准化函数

这里是这一步最核心的部分。

US 财报里常见字段：

- `ticker`
- `Company`
- `filing_date`
- `document_type`
- `fiscal_year`

CN 财报里常见字段：

- `stock_code`
- `company_name`
- `year`

我们在这里把它们映射到统一字段。

In [4]:
def clean_optional_str(value: object) -> Optional[str]:
    if value is None:
        return None
    text = str(value).strip()
    return text or None


def parse_us_filename(path: Path) -> tuple[Optional[str], Optional[str], Optional[int]]:
    parts = path.stem.split('_')
    ticker = parts[0].strip() if len(parts) >= 1 else None
    document_type = parts[1].strip() if len(parts) >= 2 else None
    filing_date = parts[2].strip() if len(parts) >= 3 else None
    report_year = None
    if filing_date:
        try:
            report_year = int(filing_date[:4])
        except ValueError:
            report_year = None
    return ticker or None, document_type or None, report_year


def load_us_document(path: Path) -> StandardizedDocument:
    """Load one US parsed filing and convert it to the unified schema.这里调动了nomalized sector所哟直接输出处理过的内容

    Args:
        path: Path to one parsed US filing JSON file.

    Returns:
        A standardized document object used by the embedding pipeline.
    """
    data = json.loads(path.read_text(encoding='utf-8'))
    meta = data.get('meta_data', {})
    sections = normalize_sections(data.get('sections', {}))
    fallback_ticker, fallback_document_type, fallback_year = parse_us_filename(path)

    ticker = clean_optional_str(meta.get('ticker')) or fallback_ticker
    company_name = clean_optional_str(meta.get('Company')) or ticker or path.stem
    filing_date = clean_optional_str(meta.get('filing_date'))
    document_type = clean_optional_str(meta.get('document_type')) or fallback_document_type or 'UNKNOWN'
    title = clean_optional_str(meta.get('title')) or path.stem
    fiscal_year_raw = meta.get('fiscal_year')
    report_year = int(fiscal_year_raw) if fiscal_year_raw not in (None, '') else fallback_year

    if report_year is None:
        raise ValueError(f'Unable to infer report_year for {path.name}')

    return StandardizedDocument(
        market='US',
        company_code=ticker or '',
        ticker=ticker,
        company_name=company_name,
        report_year=report_year,
        filing_date=filing_date,
        document_type=document_type,
        title=title,
        language='en',
        source_path=str(path.resolve()),
        source_sha256=compute_sha256(path),
        sections=sections,
    )


def load_cn_document(path: Path) -> StandardizedDocument:
    data = json.loads(path.read_text(encoding='utf-8'))
    meta = data.get('meta_data', {})
    sections = normalize_sections(data.get('sections', {}))

    return StandardizedDocument(
        market='CN',
        company_code=str(meta.get('stock_code', '')).strip(),
        ticker=None,
        company_name=str(meta.get('company_name', '')).strip(),
        report_year=int(meta.get('year')),
        filing_date=None,
        document_type='ANNUAL_REPORT',
        title=path.stem,
        language='zh',
        source_path=str(path.resolve()),
        source_sha256=compute_sha256(path),
        sections=sections,
    )

## 第 5 步：全量读取所有财报并完成标准化

到这里，我们已经有了统一 schema 和标准化函数。

这一步就是把所有 US + CN parsed filings 都读出来，全部转成 `StandardizedDocument`。

In [5]:
def load_all_standardized_documents() -> list[StandardizedDocument]:
    '''把所有的目录下的文件全都变成 StandardizedDocument 类型
    
    Arguments:
        none:没有，路径hardcode了
        
    Return:
        docs:一个全是Standardizedocument的类型的财报list'''
    docs: list[StandardizedDocument] = []

    for path in sorted(US_PARSED_DIR.glob('*.json')):
        docs.append(load_us_document(path))

    for path in sorted(CN_PARSED_DIR.glob('*.json')):
        docs.append(load_cn_document(path))

    return docs


docs = load_all_standardized_documents()

print('总文档数:', len(docs))
print('前两个样本:')
print(docs[0])
print('-' * 100)
print(docs[-1])

总文档数: 3059
前两个样本:
StandardizedDocument(market='US', company_code='AAPL', ticker='AAPL', company_name='Apple Inc.', report_year=2020, filing_date='2020-10-30', document_type='10-K', title='aapl-20200926', language='en', source_path='/Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/03_Data_Processor/US/parsed_filings/AAPL_10K_2020-10-30.json', source_sha256='51f375a6786fd2fab7140856cd94e206eb70aaad531c8bbcbf6afe7423bb1f27', sections={'item 1a': 'Item 1A. Risk Factors The following discussion of risk factors contains forward-looking statements. These risk factors may be important to understanding other statements in this Form 10-K. The following information should be read in conjunction with Part II, Item 7, “Management’s Discussion and Analysis of Financial Condition and Results of Operations” and the consolidated financial statements and accompanying notes in Part II, Item 8, “Financial Statements and Supplementary Data” of this Form 10-K. The business, financi

## 第 6 步：做一个最小检查，确认标准化后的字段是完整的

这里先检查 1 份 US 和 1 份 CN，确认：

- US 文档有 `ticker` 和 `filing_date`
- CN 文档 `ticker=None`
- 两边都能拿到 `sections`
- `sections` 里已经是清洗过的纯文本

In [9]:
sample_us = next(doc for doc in docs if doc.market == 'US')
sample_cn = next(doc for doc in docs if doc.market == 'CN')

assert sample_us.ticker is not None
assert sample_us.filing_date is not None
assert len(sample_us.sections) > 0

assert sample_cn.ticker is None
assert sample_cn.document_type == 'ANNUAL_REPORT'
assert len(sample_cn.sections) > 0

print('US sample:', sample_us.company_name, sample_us.report_year, len(sample_us.sections))
print('CN sample:', sample_cn.company_name, sample_cn.report_year, len(sample_cn.sections))

US sample: Apple Inc. 2020 21
CN sample: 平安银行 2024 10


## 第 7 步：把标准化结果保存到本地

这一步先不做 chunk，只把统一好的文档结果写成一个 `jsonl` 文件。

后面做 chunk 时，就直接从这个标准化结果出发，不需要再重新区分 US/CN 原始字段。

In [7]:
with STANDARDIZED_JSONL.open('w', encoding='utf-8') as f:
    for doc in docs:
        f.write(json.dumps(asdict(doc), ensure_ascii=False) + '\n')

print('已写入:', STANDARDIZED_JSONL)
print('文件大小（MB）:', round(STANDARDIZED_JSONL.stat().st_size / 1024 / 1024, 2))

已写入: /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/04_Embedding/chunked_data/standardized_filings.jsonl
文件大小（MB）: 1329.47


## 第 8 步：统计一下标准化后的语料情况

这里简单看一下：

- US / CN 各有多少份文档
- 每个市场的平均 section 数量

这个统计有助于我们在下一步设计 chunk 时，判断 section 粒度和数据规模。

In [8]:
us_docs = [doc for doc in docs if doc.market == 'US']
cn_docs = [doc for doc in docs if doc.market == 'CN']

avg_us_sections = round(sum(len(doc.sections) for doc in us_docs) / len(us_docs), 2)
avg_cn_sections = round(sum(len(doc.sections) for doc in cn_docs) / len(cn_docs), 2)

print('US docs:', len(us_docs), 'avg sections:', avg_us_sections)
print('CN docs:', len(cn_docs), 'avg sections:', avg_cn_sections)

US docs: 61 avg sections: 20.69
CN docs: 2998 avg sections: 9.98


## 第 9 步：开始做 chunk

现在标准化这一步已经完成了，接下来就可以把统一后的文档切成后面做 embedding 的最小检索单元。

这里的 chunk 规则固定为：

- 先按 `section` 逐段处理
- 优先按 paragraph 聚合
- 目标大小 `512 tokens`
- overlap `64 tokens`
- 超长段落先按句子拆
- 如果句子还是太长，再按 token 硬切

这样做的目的，是让后面的 embedding 不再面对整份 section，而是面对大小更稳定、语义边界更自然的 chunk。

## 第 10 步：定义 chunk 输出结构和 tokenizer

这里我们先定义 `ChunkRecord`，然后准备一个 token 计数器。

优先方案是直接使用 `BAAI/bge-m3` 的 tokenizer。这样后面切块和 embedding 的 token 口径更一致。

但为了保证 notebook 在本地环境更稳，我们也加一个 fallback：如果暂时拿不到 tokenizer，就退回一个简单的 token 估算器。

In [ ]:
CHUNKED_JSONL = OUTPUT_DIR / 'chunked_filings.jsonl'
CHUNK_TOKENS = 512
CHUNK_OVERLAP = 64
MODEL_NAME = 'BAAI/bge-m3' #beijing academy of artificial intelligence
# BAAI general embedding


@dataclass(slots=True)
class ChunkRecord:
    chunk_id: str
    market: str
    company_code: str
    ticker: Optional[str]
    company_name: str
    report_year: int
    filing_date: Optional[str]
    document_type: str
    title: str
    language: str
    source_path: str
    section_name: str
    chunk_index: int
    chunk_text: str
    token_count: int
    char_count: int


class SimpleTokenCounter:
    """A lightweight fallback tokenizer used when model tokenizer is unavailable."""

    pattern = re.compile(r"[A-Za-z0-9_]+|[\u4e00-\u9fff]|[^\s]") 
    #re 是一个正则化表达库 regular expression
    #这里匹配的是联系的英文字母和数字下划线，中文字 以及所有一般字符
    #compile表示把他变成一个可以调用的对象

    def encode(self, text: str) -> list[str]:
        '''这里输入一段text，输出tokenized list'''
        return self.pattern.findall(text or '')
    

    def count(self, text: str) -> int:
        '''这里数一共有几个token'''
        return len(self.encode(text))

    def decode(self, tokens: list[str]) -> str:
        ''' 这里输入一个token列，重新拼成一段string'''
        return ' '.join(tokens).strip()


def build_token_counter(model_name: str = MODEL_NAME):
    '''实例话一个tokenizer，要么是一个这个transforner库里auto tokenizer 
    帮你自动找到bge 的tokenizer （实际上确实能够找到），要么是用之前自己写的simple tokenizer，两个区别不是很大'''
    try:
        from transformers import AutoTokenizer

        tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)

        class HFTokenCounter:
            def encode(self, text: str) -> list[int]:
                return tokenizer.encode(text, add_special_tokens=False)

            def count(self, text: str) -> int:
                return len(self.encode(text))

            def decode(self, tokens: list[int]) -> str:
                return tokenizer.decode(tokens, skip_special_tokens=True).strip()

        print('Using HuggingFace tokenizer:', model_name)
        return HFTokenCounter()
    except Exception as e:
        print('Tokenizer fallback to SimpleTokenCounter:', repr(e))
        return SimpleTokenCounter()


token_counter = build_token_counter()
#token——counter 就是我们要用的那个tokenizer

Using HuggingFace tokenizer: BAAI/bge-m3


## 第 11 步：实现 chunk 逻辑

这里把 chunking 分成几个小函数：

- `split_paragraphs()`：先保留段落边界
- `split_sentences()`：段落太长时按句子拆
- `hard_split_by_tokens()`：句子还太长时按 token 硬切
- `chunk_section()`：把一个 section 变成多个 chunk
- `chunk_document()`：把一份标准化文档变成多个 chunk

这一步仍然只做文本处理，不做 embedding。

In [ ]:
SENTENCE_SPLIT_PATTERN = re.compile(r"(?<=[。！？!?])\s*|(?<=[.!?])\s+(?=[A-Z0-9\"'])")
# 这个是对中文来说的?<=[。！？!?])\s* 指的是 前面感叹号逗号等等加上空格。   
# 后面的是针对英文来说的表达的就是前面是。！？但是后面是英文大写开头


def split_paragraphs(text: str) -> list[str]:
    '''按paragraph 来分'''
    parts = [part.strip() for part in re.split(r'\n\s*\n+', text) if part.strip()]
    return parts or [text.strip()]


def split_sentences(text: str) -> list[str]:
    '''按照sentence 来分'''
    text = text.strip()
    if not text:
        return []
    pieces = [piece.strip() for piece in SENTENCE_SPLIT_PATTERN.split(text) if piece.strip()]
    return pieces or [text]


def token_count(text: str) -> int:
    
    return token_counter.count(text)


def decode_with_token_limit(tokens: list, max_tokens: int) -> str:
    '''保证每一个chunk 在decode 然后 在encode 的时候每一个chunk 依然在512token 范围之内'''
    window = list(tokens)
    while window:
        piece = token_counter.decode(window).strip()
        if piece and token_count(piece) <= max_tokens:
            return piece
        window = window[:-1]
    return ''


def hard_split_by_tokens(text: str, max_tokens: int = CHUNK_TOKENS, overlap: int = CHUNK_OVERLAP) -> list[str]:
    tokens = token_counter.encode(text)
    if len(tokens) <= max_tokens:
        return [text.strip()]

    out: list[str] = []
    step = max(1, max_tokens - overlap)
    for start in range(0, len(tokens), step):
        end = start + max_tokens
        piece = decode_with_token_limit(tokens[start:end], max_tokens=max_tokens)
        if piece:
            out.append(piece)
        if end >= len(tokens):
            break
    return out


def normalize_units_for_chunking(text: str, max_tokens: int = CHUNK_TOKENS) -> list[str]:
    units: list[str] = []
    for para in split_paragraphs(text):
        if token_count(para) <= max_tokens:
            units.append(para)
            continue

        for sentence in split_sentences(para):
            if token_count(sentence) <= max_tokens:
                units.append(sentence)
            else:
                units.extend(hard_split_by_tokens(sentence, max_tokens=max_tokens, overlap=CHUNK_OVERLAP))
    return [unit for unit in units if unit.strip()]


def overlap_tail(text: str, overlap: int = CHUNK_OVERLAP) -> str:
    tokens = token_counter.encode(text)
    if len(tokens) <= overlap:
        return text.strip()
    return token_counter.decode(tokens[-overlap:]).strip()


def make_chunk_record(doc: StandardizedDocument, section_name: str, chunk_index: int, chunk_text: str) -> ChunkRecord:
    chunk_id = hashlib.sha1(f'{doc.source_path}|{section_name}|{chunk_index}'.encode('utf-8')).hexdigest()
    return ChunkRecord(
        chunk_id=chunk_id,
        market=doc.market,
        company_code=doc.company_code,
        ticker=doc.ticker,
        company_name=doc.company_name,
        report_year=doc.report_year,
        filing_date=doc.filing_date,
        document_type=doc.document_type,
        title=doc.title,
        language=doc.language,
        source_path=doc.source_path,
        section_name=section_name,
        chunk_index=chunk_index,
        chunk_text=chunk_text,
        token_count=token_count(chunk_text),
        char_count=len(chunk_text),
    )


def append_chunk_text(out: list[ChunkRecord], doc: StandardizedDocument, section_name: str, chunk_text: str, max_tokens: int = CHUNK_TOKENS, overlap: int = CHUNK_OVERLAP) -> None:
    cleaned = chunk_text.strip()
    if not cleaned:
        return
    pieces = [cleaned] if token_count(cleaned) <= max_tokens else hard_split_by_tokens(cleaned, max_tokens=max_tokens, overlap=overlap)
    for piece in pieces:
        normalized_piece = piece.strip()
        if normalized_piece:
            out.append(make_chunk_record(doc, section_name, len(out), normalized_piece))


def chunk_section(doc: StandardizedDocument, section_name: str, text: str, max_tokens: int = CHUNK_TOKENS, overlap: int = CHUNK_OVERLAP) -> list[ChunkRecord]:
    units = normalize_units_for_chunking(text, max_tokens=max_tokens)
    if not units:
        return []

    out: list[ChunkRecord] = []
    current = units[0]

    for unit in units[1:]:
        candidate = f'{current}\n\n{unit}'.strip()
        if token_count(candidate) <= max_tokens:
            current = candidate
            continue

        append_chunk_text(out, doc, section_name, current, max_tokens=max_tokens, overlap=overlap)
        tail = overlap_tail(current, overlap=overlap)
        merged = f'{tail}\n\n{unit}'.strip() if tail else unit
        current = merged if token_count(merged) <= max_tokens else unit

    append_chunk_text(out, doc, section_name, current, max_tokens=max_tokens, overlap=overlap)
    return out


def chunk_document(doc: StandardizedDocument, max_tokens: int = CHUNK_TOKENS, overlap: int = CHUNK_OVERLAP) -> list[ChunkRecord]:
    records: list[ChunkRecord] = []
    for section_name, section_text in doc.sections.items():
        records.extend(chunk_section(doc, section_name, section_text, max_tokens=max_tokens, overlap=overlap))
    return records


def chunk_all_documents(documents: list[StandardizedDocument], max_tokens: int = CHUNK_TOKENS, overlap: int = CHUNK_OVERLAP) -> list[ChunkRecord]:
    all_chunks: list[ChunkRecord] = []
    for doc in documents:
        all_chunks.extend(chunk_document(doc, max_tokens=max_tokens, overlap=overlap))
    return all_chunks


## 第 12 步：先用 1 份 US + 1 份 CN 做小样本检查

在真正全量切块之前，先确认这几个最关键的条件：

- chunk 数大于 0
- 每个 chunk 都不超过目标 token 上限
- section_name、chunk_index、chunk_text 都是完整的
- 中英文文档都能切出合理结果

In [ ]:
sample_us_chunks = chunk_document(sample_us)
sample_cn_chunks = chunk_document(sample_cn)

assert sample_us_chunks, 'US sample 没有切出 chunk'
assert sample_cn_chunks, 'CN sample 没有切出 chunk'
assert max(chunk.token_count for chunk in sample_us_chunks) <= CHUNK_TOKENS
assert max(chunk.token_count for chunk in sample_cn_chunks) <= CHUNK_TOKENS

print('US sample chunk 数:', len(sample_us_chunks))
print('CN sample chunk 数:', len(sample_cn_chunks))
print('US 第一个 chunk:')
print(sample_us_chunks[0])
print(sample_us_chunks[0].chunk_text[:400])
print('-' * 100)
print('CN 第一个 chunk:')
print(sample_cn_chunks[0])
print(sample_cn_chunks[0].chunk_text[:400])

## 第 13 步：全量做 chunk，并写出 `chunked_filings.jsonl`

这里会把前面标准化后的所有文档全部切成 chunk，然后落盘。

后面做 embedding 的时候，就不再面对整份 section，而是直接面对 `chunked_filings.jsonl` 里的每一个 chunk。

In [ ]:
all_chunks = chunk_all_documents(docs)

with CHUNKED_JSONL.open('w', encoding='utf-8') as f:
    for chunk in all_chunks:
        f.write(json.dumps(asdict(chunk), ensure_ascii=False) + '\n')

print('总 chunk 数:', len(all_chunks))
print('已写入:', CHUNKED_JSONL)
print('文件大小（MB）:', round(CHUNKED_JSONL.stat().st_size / 1024 / 1024, 2))

## 第 14 步：统计 chunk 后的结果

这一格主要是帮助你判断后面 embedding 的规模：

- 总 chunk 数
- US / CN 各有多少 chunk
- 平均每份文档能切出多少 chunk
- 最大 chunk token 数是否符合预期

In [ ]:
us_chunks = [chunk for chunk in all_chunks if chunk.market == 'US']
cn_chunks = [chunk for chunk in all_chunks if chunk.market == 'CN']

avg_chunks_per_doc = round(len(all_chunks) / len(docs), 2)
max_chunk_tokens = max(chunk.token_count for chunk in all_chunks) if all_chunks else 0

print('US chunks:', len(us_chunks))
print('CN chunks:', len(cn_chunks))
print('Avg chunks per doc:', avg_chunks_per_doc)
print('Max chunk tokens:', max_chunk_tokens)

## 下一步：开始做 embedding

现在你已经有两层中间结果：

- `standardized_filings.jsonl`
- `chunked_filings.jsonl`

后面做 embedding 时，直接读取 `chunked_filings.jsonl` 即可。下一步的核心工作就是：

- 定义 `embed_texts()`
- 用 `BAAI/bge-m3` 对每个 chunk 生成向量
- 把向量和 chunk metadata 一起存下来
- 给 query 做同样的 embedding，再做相似度检索